# Run 2a Results Analysis
Adapted from `run1_results_analysis.ipynb` (Notebook 3)

## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 60)

RUN2A_OUTPUT  = 'outputs/run2a_results_final.csv'
TAXONOMY_PATH = 'outputs/taxonomy_master.csv'

df = pd.read_csv(RUN2A_OUTPUT)

# Fix confidence type immediately after load
#df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')
conf_map = {'high': 1.0, 'medium': 0.5, 'low': 0.0}
df['confidence'] = df['confidence'].map(conf_map)

print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
print(f'Confidence non-null: {df["confidence"].notna().sum()} / {len(df)}')

In [ ]:
print(df[['confidence', 'reason']].head(5).to_string())

## 2. Taxonomy Join & Accuracy Eval
- Drops blank GT cols from Run 2a output and rebuilds from taxonomy
- Resolves `'mixed'` spend_type via `leans` / `manual_spend_override`
- Avoids column collisions (taxonomy also has `confidence`)

In [ ]:
tax = pd.read_csv(TAXONOMY_PATH)

# --- GT lookup (keyed on ground_truth) ---
gt_lookup = tax.set_index('base_category_key')[[
    'spend_type', 'leans', 'manual_spend_override', 'category_group', 'category_type'
]].rename(columns={
    'spend_type':            'gt_spend_type_raw',
    'leans':                 'gt_leans',
    'manual_spend_override': 'gt_spend_override',
    'category_group':        'gt_category_group',
    'category_type':         'gt_category_type',
})

# Drop any existing GT cols (blank in Run 2a output) and re-join from taxonomy
drop_cols = ['gt_spend_type','gt_spend_type_raw','gt_leans','gt_spend_override',
             'gt_category_group','gt_category_type','gt_group','gt_cat_type',
             'l1_match','l2_match']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
df = df.merge(gt_lookup, left_on='ground_truth', right_index=True, how='left')

# --- Resolve mixed GT spend_type ---
def resolve_spend_type(row):
    raw = str(row.get('gt_spend_type_raw', '')).lower().strip()
    if raw not in ('mixed', 'nan', ''):
        return raw
    override = str(row.get('gt_spend_override', '')).lower().strip()
    if override in ('spend', 'non_spend'):
        return override
    leans = str(row.get('gt_leans', '')).lower().strip()
    if leans in ('spend', 'non_spend'):
        return leans
    return raw  # leave as mixed/unknown if unresolvable

df['gt_spend_type'] = df.apply(resolve_spend_type, axis=1)

# --- L1: spend_type ---
l1_mask = df['gt_spend_type'].isin(['spend', 'non_spend'])
df['l1_match'] = df['llm_spend_type'] == df['gt_spend_type']

# --- L2: category_group (spend) or category_type (non_spend) ---
# Use only the two columns needed — avoid confidence collision
pred_lookup = tax.set_index('base_category_key')[['category_group', 'category_type']].rename(columns={
    'category_group': 'pred_category_group',
    'category_type':  'pred_category_type',
})
df = df.merge(pred_lookup, left_on='base_category_key', right_index=True, how='left')

def l2_match(row):
    st = row.get('gt_spend_type', '')
    if st == 'spend':
        return row.get('pred_category_group') == row.get('gt_category_group')
    elif st == 'non_spend':
        return row.get('pred_category_type') == row.get('gt_category_type')
    return False

df['l2_match'] = df.apply(l2_match, axis=1)

# --- L3: exact key ---
df['l3_match'] = df['ground_truth'] == df['base_category_key']

# --- Summary ---
print('=== Accuracy Summary ===')
print(f'L1 spend_type (resolved, {l1_mask.sum()} evaluable rows): {df.loc[l1_mask, "l1_match"].mean():.1%}')
print(f'L2 group/type:                                            {df["l2_match"].mean():.1%}')
print(f'L3 exact key:                                             {df["l3_match"].mean():.1%}')
print(f'Unresolvable GT spend_type (excl. from L1):               {(~l1_mask).sum()}')
print(f'\nGT spend_type breakdown:')
print(df['gt_spend_type'].value_counts(dropna=False))

## 3. Mismatch Overview

In [ ]:
mismatches = df[~df['l3_match']].copy()
matches    = df[ df['l3_match']].copy()

l1_wrong = mismatches[~mismatches['l1_match'] & l1_mask.reindex(mismatches.index, fill_value=False)]
l1_right = mismatches[ mismatches['l1_match']]

print(f'Matches:    {len(matches):,} ({len(matches)/len(df):.1%})')
print(f'Mismatches: {len(mismatches):,} ({len(mismatches)/len(df):.1%})')
print(f'  Wrong spend_type:      {len(l1_wrong)}  ({len(l1_wrong)/len(mismatches):.0%} of mismatches)')
print(f'  Right type, wrong key: {len(l1_right)}  ({len(l1_right)/len(mismatches):.0%} of mismatches)')

cols = ['original_description', 'merchant_name', 'ground_truth',
        'base_category_key', 'reason', 'confidence', 'llm_spend_type', 'gt_spend_type']
display(mismatches[cols].sort_values('ground_truth').head(40))

## 4. Top Mismatch Pairs

In [ ]:
pair_counts = (
    mismatches
    .groupby(['ground_truth', 'base_category_key'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
print('=== Top 15 Mismatch Pairs ===')
display(pair_counts.head(15))

## 5. Confusion Matrix (top keys)

In [ ]:
top_keys = df['ground_truth'].value_counts().head(20).index
cm_df = df[df['ground_truth'].isin(top_keys) | df['base_category_key'].isin(top_keys)]
cm = pd.crosstab(cm_df['ground_truth'], cm_df['base_category_key'])

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.5})
ax.set_title('Run 2a Confusion Matrix (top 20 GT keys)', fontsize=13)
ax.set_xlabel('Predicted key')
ax.set_ylabel('Ground truth key')
plt.tight_layout()
plt.show()

## 6. Confidence vs Accuracy

In [ ]:
conf_acc = (
    df.assign(conf_bin=pd.cut(df['confidence'], bins=[0,.1,.2,.3,.4,.5,.6,.7,.8,.9,1.0],
                              labels=[f'{i*10}-{i*10+10}%' for i in range(10)]))
    .groupby('conf_bin', observed=True)['l3_match']
    .agg(['mean','count'])
    .reset_index()
    .rename(columns={'conf_bin':'confidence_bin','mean':'l3_accuracy','count':'n'})
)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(conf_acc['confidence_bin'], conf_acc['n'], alpha=0.3, color='steelblue')
ax2.plot(conf_acc['confidence_bin'], conf_acc['l3_accuracy'], 'o-', color='crimson')
ax1.set_xlabel('Confidence bin')
ax1.set_ylabel('Row count', color='steelblue')
ax2.set_ylabel('L3 accuracy', color='crimson')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.set_xticklabels(conf_acc['confidence_bin'], rotation=45, ha='right')
ax1.set_title('Confidence Distribution vs L3 Accuracy — Run 2a')
fig.tight_layout()
plt.show()

display(conf_acc.assign(l3_accuracy=conf_acc['l3_accuracy'].map('{:.1%}'.format)))
print(f'\nOverall avg confidence: {df["confidence"].mean():.2f}  |  non-null: {df["confidence"].notna().sum()}')

## 7. Provider Breakdown

In [ ]:
prov = df.groupby('provider_name').agg(
    n        = ('l3_match', 'count'),
    l1_acc   = ('l1_match', 'mean'),
    l3_acc   = ('l3_match', 'mean'),
    avg_conf = ('confidence', 'mean'),
).sort_values('l3_acc', ascending=False).reset_index()

display(prov.assign(
    l1_acc   = prov['l1_acc'].map('{:.1%}'.format),
    l3_acc   = prov['l3_acc'].map('{:.1%}'.format),
    avg_conf = prov['avg_conf'].map(lambda x: f'{x:.2f}' if pd.notna(x) else 'n/a'),
))

low_provs = prov[prov['l3_acc'] < 0.5]['provider_name'].tolist()
if low_provs:
    print(f'\n⚠️  Providers < 50% L3: {low_provs}')
    display(mismatches[mismatches['provider_name'].isin(low_provs)][cols].head(20))

## 8. Reason Field — Is the LLM Using Hints?

In [ ]:
# Actual hint coverage — was a hint present in the data?
df['hint_biller']     = df['biller_category'].ne('none')
df['hint_mcc']        = df['mcc_category'].ne('none')
df['hint_spend_type'] = df['dt_hint'].ne('none')
df['hint_merchant']   = df['merchant_name'].notna() & df['merchant_name'].ne('')
df['hint_regex']      = df['regex_category_type'].notna() & df['regex_category_type'].ne('none')

hint_cols = ['hint_biller', 'hint_mcc', 'hint_spend_type', 'hint_merchant', 'hint_regex']
mismatches = df[~df['l3_match']].copy()

summary = pd.DataFrame({
    'all_rows':   df[hint_cols].mean(),
    'mismatches': mismatches[hint_cols].mean(),
    'matches':    df[df['l3_match']][hint_cols].mean(),
}).rename(index=lambda x: x.replace('hint_', ''))

display(summary.applymap('{:.1%}'.format))

## 9. Spend_type Mismatch Deep Dive

In [ ]:
# Recompute l1_wrong/l1_right against refreshed mismatches
l1_wrong = mismatches[~mismatches['l1_match'] & l1_mask.reindex(mismatches.index, fill_value=False)]
l1_right = mismatches[ mismatches['l1_match']]

if len(l1_wrong) > 0:
    print(f'=== L1 Wrong ({len(l1_wrong)} rows) ===')
    display(l1_wrong.groupby(['gt_spend_type', 'llm_spend_type']).size().reset_index(name='count'))

    print('\nSample: GT=non_spend, predicted=spend')
    display(l1_wrong[l1_wrong['gt_spend_type']=='non_spend'][
        ['original_description','merchant_name','ground_truth','llm_spend_type','reason']
    ].head(10))

    print('\nSample: GT=spend, predicted=non_spend')
    display(l1_wrong[l1_wrong['gt_spend_type']=='spend'][
        ['original_description','merchant_name','ground_truth','llm_spend_type','reason']
    ].head(10))
else:
    print('No L1 wrong rows (or GT spend_type unresolvable for all mismatches)')

## 10. Summary

In [ ]:
top_pair = (f"{pair_counts.iloc[0]['ground_truth']} → {pair_counts.iloc[0]['base_category_key']}"
            f" (n={pair_counts.iloc[0]['count']})") if len(pair_counts) else 'N/A'

print('=' * 55)
print('RUN 2a ANALYSIS SUMMARY')
print('=' * 55)
print(f'Total rows:              {len(df)}')
print(f'L3 accuracy:             {df["l3_match"].mean():.1%}')
print(f'L2 accuracy:             {df["l2_match"].mean():.1%}')
print(f"L2a category_type match:  {df['gt_category_type'].eq(df['pred_category_type']).mean()*100:.1f}%")
print(f"L2b category_group match: {df['gt_category_group'].eq(df['pred_category_group']).mean()*100:.1f}%")
print(f'L1 accuracy (fixed):     {df.loc[l1_mask, "l1_match"].mean():.1%}  ({l1_mask.sum()} evaluable)')
print(f'Mismatches:              {len(mismatches)}')
print(f'  Wrong spend_type:      {len(l1_wrong)} ({len(l1_wrong)/max(len(mismatches),1):.0%})')
print(f'  Right type, wrong key: {len(l1_right)} ({len(l1_right)/max(len(mismatches),1):.0%})')
print(f'Avg confidence:          {df["confidence"].mean():.2f}')
print(f'Top mismatch pair:       {top_pair}')
print('=' * 55)

In [ ]:
print(df['confidence'].dtype)
print(df['confidence'].value_counts(dropna=False).head(10))

In [ ]:
print(df['reason'].head(3).tolist())

In [ ]:
# Look at rows where L3 matches but L2 doesn't
l3_yes_l2_no = df[df['match'] & ~df['l2_match']]
print(f"L3 correct but L2 wrong: {len(l3_yes_l2_no)}")
print(l3_yes_l2_no[['ground_truth', 'base_category_key', 
                      'gt_category_group', 'gt_category_type',
                      'pred_category_group', 'pred_category_type']].head(10).to_string())

In [ ]:
print(df[['dt_hint', 'dt_rule', 'biller_category', 'mcc_category']].value_counts(dropna=False).head(10))

In [ ]:
# 1. Raw coverage
print("Biller codes present:", df['biller_category'].ne('none').sum())
print("Regex hits present:", df['regex_category_type'].ne('none').sum())

# 2. Sample of non-none values if any exist
print("\nBiller sample:", df[df['biller_category'].ne('none')]['biller_category'].value_counts().head(5))
print("Regex sample:", df[df['regex_category_type'].ne('none')]['regex_category_type'].value_counts().head(5))

In [ ]:
# How many rows are non_spend per DT?
print(df['dt_hint'].value_counts())

# Of those non_spend rows, what did regex return?
ns_rows = df[df['dt_hint'] == 'non_spend']
print(f"\nNon-spend rows: {len(ns_rows)}")
print(ns_rows['regex_category_type'].value_counts(dropna=False))

In [ ]:
ns_rows = df[df['dt_hint'] == 'non_spend'][
    ['original_description', 'ground_truth', 'regex_category_type']
].to_string()
print(ns_rows)

In [ ]:
from ipywidgets import interact, IntSlider, ToggleButtons

def review_row(subset, idx):
    data = mismatches if subset == 'Mismatches only' else df
    row = data.iloc[idx]
    status = "✅ MATCH" if row['match'] else "❌ MISMATCH"
    print(f"{'='*60}")
    print(f"Row {idx} | {status} | Confidence: {row['confidence']}")
    print(f"{'='*60}")
    print(f"Provider:     {row['provider_name']}")
    print(f"Description:  {row['original_description']}")
    print(f"Merchant:     {row['merchant_name']}")
    print(f"Amount:       {row['amount']}")
    print(f"Txn type:     {row['transaction_type']}")
    print(f"")
    print(f"── Hints ──────────────────────────────────────────")
    print(f"DT hint:      {row['dt_hint']} ({row['dt_rule']})")
    print(f"Biller:       {row['biller_category']}")
    print(f"MCC:          {row['mcc_category']}")
    print(f"Regex:        {row['regex_category_type']}")
    print(f"")
    print(f"── LLM Output ─────────────────────────────────────")
    print(f"spend_type:   {row['llm_spend_type']}")
    print(f"Predicted:    {row['base_category_key']} ({row['category_key_method']})")
    print(f"Ground truth: {row['ground_truth']}")
    print(f"Reason:       {row['reason']}")
    if pd.notna(row.get('flag_type')):
        print(f"Flag:         {row['flag_type']} — {row['flag_detail']}")

def make_widget(subset):
    data = mismatches if subset == 'Mismatches only' else df
    interact(
        review_row,
        subset=ToggleButtons(options=['All rows', 'Mismatches only'], value=subset),
        idx=IntSlider(min=0, max=len(data)-1, step=1, value=0)
    )

interact(
    make_widget,
    subset=ToggleButtons(options=['All rows', 'Mismatches only'], value='All rows')
)

In [ ]:
from ipywidgets import interact, IntSlider, ToggleButtons
import ipywidgets as widgets

subset_toggle = ToggleButtons(options=['All rows', 'Mismatches only'], value='All rows')
idx_slider = IntSlider(min=0, max=len(df)-1, step=1, value=0)

def update_slider(subset):
    data = mismatches if subset == 'Mismatches only' else df
    idx_slider.max = len(data) - 1
    idx_slider.value = 0

def review_row(subset, idx):
    data = mismatches if subset == 'Mismatches only' else df
    row = data.iloc[idx]
    status = "✅ MATCH" if row['match'] else "❌ MISMATCH"
    print(f"{'='*60}")
    print(f"Row {idx} | {status} | Confidence: {row['confidence']}")
    print(f"{'='*60}")
    print(f"Provider:     {row['provider_name']}")
    print(f"Description:  {row['original_description']}")
    print(f"Merchant:     {row['merchant_name']}")
    print(f"Amount:       {row['amount']}")
    print(f"Txn type:     {row['transaction_type']}")
    print(f"")
    print(f"── Hints ──────────────────────────────────────────")
    print(f"DT hint:      {row['dt_hint']} ({row['dt_rule']})")
    print(f"Biller:       {row['biller_category']}")
    print(f"MCC:          {row['mcc_category']}")
    print(f"Regex:        {row['regex_category_type']}")
    print(f"")
    print(f"── LLM Output ─────────────────────────────────────")
    print(f"spend_type:   {row['llm_spend_type']}")
    print(f"Predicted:    {row['base_category_key']} ({row['category_key_method']})")
    print(f"Ground truth: {row['ground_truth']}")
    print(f"Reason:       {row['reason']}")
    if pd.notna(row.get('flag_type')):
        print(f"Flag:         {row['flag_type']} — {row['flag_detail']}")

subset_toggle.observe(lambda change: update_slider(change['new']), names='value')

interact(review_row, subset=subset_toggle, idx=idx_slider)

In [ ]:
from ipywidgets import interact, IntSlider, ToggleButtons, Dropdown, HBox, VBox, Output
import ipywidgets as widgets

# ── Build filter options ───────────────────────────────────────────────────────
spend_opts   = ['Any', 'spend', 'non_spend']
gt_opts      = ['Any'] + sorted(df['ground_truth'].dropna().unique().tolist())
pred_opts    = ['Any'] + sorted(df['base_category_key'].dropna().unique().tolist())

subset_toggle  = ToggleButtons(options=['All rows', 'Mismatches only'], value='All rows')
spend_dd       = Dropdown(options=spend_opts, value='Any', description='Spend type:')
gt_dd          = Dropdown(options=gt_opts,    value='Any', description='GT key:')
pred_dd        = Dropdown(options=pred_opts,  value='Any', description='Predicted:')
idx_slider     = IntSlider(min=0, max=len(df)-1, step=1, value=0, description='Row:')
out            = Output()

def get_filtered():
    data = mismatches if subset_toggle.value == 'Mismatches only' else df
    if spend_dd.value != 'Any':
        data = data[data['llm_spend_type'] == spend_dd.value]
    if gt_dd.value != 'Any':
        data = data[data['ground_truth'] == gt_dd.value]
    if pred_dd.value != 'Any':
        data = data[data['base_category_key'] == pred_dd.value]
    return data.reset_index(drop=True)

def update_slider(*args):
    data = get_filtered()
    idx_slider.max = max(len(data) - 1, 0)
    idx_slider.value = 0
    render()

def render(*args):
    data = get_filtered()
    out.clear_output(wait=True)
    with out:
        if len(data) == 0:
            print("No rows match the current filters.")
            return
        idx = min(idx_slider.value, len(data) - 1)
        row = data.iloc[idx]
        status = "✅ MATCH" if row['match'] else "❌ MISMATCH"
        print(f"Showing {idx + 1} of {len(data)} rows")
        print(f"{'='*60}")
        print(f"Row {idx} | {status} | Confidence: {row['confidence']}")
        print(f"{'='*60}")
        print(f"Provider:     {row['provider_name']}")
        print(f"Description:  {row['original_description']}")
        print(f"Merchant:     {row['merchant_name']}")
        print(f"Amount:       {row['amount']}")
        print(f"Txn type:     {row['transaction_type']}")
        print(f"")
        print(f"── Hints ──────────────────────────────────────────")
        print(f"DT hint:      {row['dt_hint']} ({row['dt_rule']})")
        print(f"Biller:       {row['biller_category']}")
        print(f"MCC:          {row['mcc_category']}")
        print(f"Regex:        {row['regex_category_type']}")
        print(f"")
        print(f"── LLM Output ─────────────────────────────────────")
        print(f"spend_type:   {row['llm_spend_type']}")
        print(f"Predicted:    {row['base_category_key']} ({row['category_key_method']})")
        print(f"Ground truth: {row['ground_truth']}")
        print(f"Reason:       {row['reason']}")
        if pd.notna(row.get('flag_type')):
            print(f"Flag:         {row['flag_type']} — {row['flag_detail']}")

# ── Wire observers ─────────────────────────────────────────────────────────────
for w in [subset_toggle, spend_dd, gt_dd, pred_dd]:
    w.observe(update_slider, names='value')
idx_slider.observe(render, names='value')

# ── Layout & display ───────────────────────────────────────────────────────────
display(VBox([
    subset_toggle,
    HBox([spend_dd, gt_dd, pred_dd]),
    idx_slider,
    out
]))
update_slider()

In [ ]:
# 1. INTERNAL_TRANSFER → ROUND_UP cluster
cluster = df[(df['ground_truth'] == 'INTERNAL_TRANSFER') & 
             (df['base_category_key'] == 'ROUND_UP')]
print(f"=== INTERNAL_TRANSFER → ROUND_UP (n={len(cluster)}) ===")
print(cluster[['original_description', 'merchant_name', 'amount', 
               'dt_hint', 'mcc_category', 'llm_spend_type', 'reason']].to_string())

# 2. L1 wrong rows (spend_type mismatch)
print(f"\n=== L1 Wrong (n={len(l1_wrong)}) ===")
print(l1_wrong[['original_description', 'merchant_name', 'ground_truth',
                 'llm_spend_type', 'dt_hint', 'reason']].to_string())

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/run2a_results.csv")  # adjust filename to match yours
print(f"Total rows: {len(df)}")
print(f"Last few rows:")
print(df.tail())